In [1]:
import os
import glob
import pandas as pd

def inspect_raw_cataluminescence(file_path):
    """
    Dynamically locates the table header, loads the CSV, and outputs 
    shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Check if the file has Agilent metadata at the top
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line or 'timestamp' in line.lower():
                header_row_index = idx
                break
                
    # Load the dataset from the detected header
    df = pd.read_csv(file_path, skiprows=header_row_index)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON CATALUMINESCENCE FOLDER
# ==========================================
cat_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence"
csv_files = glob.glob(os.path.join(cat_path, "*.csv"))

print(f"Found {len(csv_files)} files in cataluminiscence folder. Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    # We will inspect all files, focusing especially on the two raw files
    try:
        df = inspect_raw_cataluminescence(file_path)
        
        print(f"📄 File {i}: {file_name}")
        print(f"   • Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"   • Columns: {list(df.columns)}")
        print(f"   • Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   --- Baseline Data Summary (First 3 Numeric Columns) ---")
            print(df[numeric_cols[:3]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"❌ Could not read {file_name}: {e}\n")

Found 3 files in cataluminiscence folder. Running inspection...

📄 File 1: 1781339209_2 2025-11-21 0.csv
   • Shape: 2770 rows × 4 columns
   • Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101Time (Sec)', '101 (°C)']
   • Missing Values: 0 total nulls

   --- Baseline Data Summary (First 3 Numeric Columns) ---
      Scan Number    101 (°C)
mean  1385.500000  297.906934
min      1.000000   23.402155
max   2770.000000  427.728817
std    799.774447  150.753252


📄 File 2: 1781339239_2 2025-11-21 0.csv
   • Shape: 2770 rows × 4 columns
   • Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101Time (Sec)', '101 (°C)']
   • Missing Values: 0 total nulls

   --- Baseline Data Summary (First 3 Numeric Columns) ---
      Scan Number    101 (°C)
mean  1385.500000  297.906934
min      1.000000   23.402155
max   2770.000000  427.728817
std    799.774447  150.753252


📄 File 3: 20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv
   • Shape: 10497 rows × 5 columns
   • Columns: ['timestamp', 

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. ADAPTIVE DATA LOADER (RAW vs. CLEANED)
# =========================================================
def load_cataluminescence_file(file_path):
    """
    Automatically detects whether the file is a raw Agilent hardware log
    or a preprocessed '_cleaned.csv' file, standardizing the time and feature columns.
    """
    file_name = os.path.basename(file_path)
    
    # Check if the file is already cleaned or raw Agilent
    if "cleaned" in file_name.lower():
        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()
        time_col = 'timestamp' if 'timestamp' in df.columns else df.index
        feature_col = 'ch_00'
    else:
        # Scan for raw Agilent header row
        header_idx = 0
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for idx, line in enumerate(f):
                if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee', '101Time']):
                    header_idx = idx
                    break
        df = pd.read_csv(file_path, skiprows=header_idx)
        df.columns = df.columns.str.strip()
        df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
        
        # Standardize timestamp and feature columns
        time_col = [col for col in df.columns if 'time' in col.lower() or 'swee' in col.lower()][0]
        feature_col = [col for col in df.columns if '101' in col and '°c' in col.lower()][0]

    # Standardize column names for clean downstream processing
    df['Timestamp_Clean'] = df[time_col]
    df['Cataluminescence_Signal'] = pd.to_numeric(df[feature_col], errors='coerce')
    
    return df, 'Cataluminescence_Signal'

# =========================================================
# 2. STATISTICAL ENGINE
# =========================================================
def compute_cataluminescence_statistics(df, feature_col, file_name):
    """
    Computes univariate statistics for the long-duration cataluminescence signal.
    """
    series = df[feature_col].dropna()
    
    print(f"================================================================")
    print(f" STATISTICAL SUMMARY: {file_name}")
    print(f"================================================================")
    
    stats_dict = {
        'Total Scans': len(series),
        'Mean': series.mean(),
        'Std Dev': series.std(),
        'Min': series.min(),
        '25% (Q1)': series.quantile(0.25),
        '50% (Median)': series.median(),
        '75% (Q3)': series.quantile(0.75),
        'Max': series.max(),
        'IQR': series.quantile(0.75) - series.quantile(0.25),
        'Skewness': series.skew(),
        'Kurtosis': series.kurtosis(),
        'Null Count': df[feature_col].isnull().sum()
    }
    
    stats_df = pd.DataFrame.from_dict(stats_dict, orient='index', columns=['Value'])
    print(stats_df.round(4))
    print("================================================================\n")

# =========================================================
# 3. INTERACTIVE PLOTLY VISUALIZATIONS (PER CSV FILE)
# =========================================================
def plot_whole_dataset_cat(df, feature_col, file_name):
    """
    Renders the long-duration cataluminescence signal trend across the entire run.
    """
    fig = px.line(
        df,
        x='Timestamp_Clean',
        y=feature_col,
        title=f"Whole-Dataset Cataluminescence Signal Trend — {file_name}",
        labels={feature_col: 'Signal Intensity / Temp (°C)', 'Timestamp_Clean': 'Timestamp / Elapsed Time'}
    )
    fig.update_layout(
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True), type='category'),
        height=450,
        width=1200
    )
    fig.show()

def plot_individual_feature_cat(df, feature_col, file_name):
    """
    Renders a 1x2 subplot pairing the signal time-series with an interactive
    statistical box plot to inspect baseline stability and outlier boundaries.
    """
    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.75, 0.25],
        subplot_titles=[f"Signal Time-Series Trend", f"Outlier Distribution Box Plot"],
        horizontal_spacing=0.08
    )
    
    # Left Column: Time-Series Signal
    fig.add_trace(
        go.Scatter(
            x=df['Timestamp_Clean'],
            y=df[feature_col],
            mode='lines',
            name="Signal",
            line=dict(color='#1f77b4', width=1.5),
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Right Column: Statistical Box Plot (Outlier Detection Boundaries)
    fig.add_trace(
        go.Box(
            y=df[feature_col].dropna(),
            name="Distribution",
            marker_color='#1f77b4',
            boxpoints='outliers',
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.update_yaxes(title_text="Signal Intensity (°C)", row=1, col=1)
    fig.update_layout(
        title_text=f"Individual Feature Inspection — {file_name}",
        height=450,
        width=1200,
        showlegend=False
    )
    fig.show()

# =========================================================
# 4. EXECUTION LOOP: PROCESS EACH CSV SEPARATELY
# =========================================================
cat_dir = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence"
csv_files = sorted(glob.glob(os.path.join(cat_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV files in cataluminiscence folder. Processing each separately...\n")

for i, sample_file in enumerate(csv_files, 1):
    file_name = os.path.basename(sample_file)
    print(f"\n======== PROCESSING FILE {i}/{len(csv_files)}: {file_name} ========")
    
    try:
        # 1. Adaptively load raw vs. cleaned data
        df_cat, feature_name = load_cataluminescence_file(sample_file)
        
        # 2. Compute univariate statistical metrics
        compute_cataluminescence_statistics(df_cat, feature_name, file_name)
        
        # 3. Generate Whole-Dataset Time-Series Plot
        plot_whole_dataset_cat(df_cat, feature_name, file_name)
        
        # 4. Generate Individual Feature Box Plot & Trend Subplots
        plot_individual_feature_cat(df_cat, feature_name, file_name)
        
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")

Found 3 CSV files in cataluminiscence folder. Processing each separately...


======== PROCESSING FILE 1/3: 1781339209_2 2025-11-21 0.csv ========
 STATISTICAL SUMMARY: 1781339209_2 2025-11-21 0.csv
                  Value
Total Scans   2770.0000
Mean           297.9069
Std Dev        150.7533
Min             23.4022
25% (Q1)       185.5085
50% (Median)   379.2873
75% (Q3)       418.2441
Max            427.7288
IQR            232.7356
Skewness        -0.9122
Kurtosis        -0.7998
Null Count       0.0000




======== PROCESSING FILE 2/3: 1781339239_2 2025-11-21 0.csv ========
 STATISTICAL SUMMARY: 1781339239_2 2025-11-21 0.csv
                  Value
Total Scans   2770.0000
Mean           297.9069
Std Dev        150.7533
Min             23.4022
25% (Q1)       185.5085
50% (Median)   379.2873
75% (Q3)       418.2441
Max            427.7288
IQR            232.7356
Skewness        -0.9122
Kurtosis        -0.7998
Null Count       0.0000




======== PROCESSING FILE 3/3: 20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv ========
 STATISTICAL SUMMARY: 20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv
                   Value
Total Scans   10497.0000
Mean            337.4393
Std Dev         137.9531
Min              53.7221
25% (Q1)        227.3286
50% (Median)    416.0931
75% (Q3)        427.4213
Max             469.4448
IQR             200.0927
Skewness         -1.0526
Kurtosis         -0.5927
Null Count        0.0000

